In [0]:
from scr.endpoints import list_keys_all, list_keys_changes, items_many, next_offset
from scr.connection import APIClient
from scr.loaders import fetch_all_keys_to_bronze
import requests
import time
import json

BASE = "https://productapi-synkka.gs1.fi"
EMAIL = dbutils.secrets.get("gs1-kv", "email")
PASSWORD = dbutils.secrets.get("gs1-kv", "password")
GLN = dbutils.secrets.get("gs1-kv", "gln")

account   = "gs1datalake"      # storage accountin nimi
container = "datalake"         # containerin nimi
access_key = "5kEehpTCdCfzcmwOzmq+w4iaiFeMGnfV2OCaxQlGut2kb65ItOD4QDWapkmcT/NI4t8sLaOFAbjL+AStaIorWg=="

# kerrotaan Sparkin asetuksissa, miten mennään storageen
spark.conf.set(f"fs.azure.account.key.{account}.dfs.core.windows.net", access_key)

# muodostetaan polku, johon kirjoitetaan/lukemaan
DELTA_KEYS = f"abfss://{container}@{account}.dfs.core.windows.net/gs1/bronze/public_item_sync"
DELTA_PRODUCTS = f"abfss://{container}@{account}.dfs.core.windows.net/gs1/silver/catalogue_items"

KEY_BATCH_SIZE = 90000
SINCE_ISO = "2025-09-10T07:03:26.1655687Z"

client = APIClient(BASE, EMAIL, PASSWORD, GLN)
client.authenticate(verbose=True)   

all_keys = fetch_all_keys_to_bronze(spark, client, DELTA_KEYS, batch_size=KEY_BATCH_SIZE)
print(f"Avaimien (All) nouto onnistui, avaimia talletettu: {all_keys}")

time.sleep(1.2)

# Changes → ensin kerrotaan mistä alkaen haetaan
print(f"Aloitetaan avaimien (Changes) nouto alkaen: {SINCE_ISO.split("T", 1)[0]}")
resp_chg = list_keys_changes(client, batch_size=KEY_BATCH_SIZE, since=SINCE_ISO)
chg_items = resp_chg.get("Items") or []
print(f"Avaimien (Changes) nouto onnistui, avaimia haettu: {len(chg_items)}")

time.sleep(1.2)

# Testaa /Many muutamalle Id:lle (ensin changesistä, muuten all:sta)
sample_ids = [it.get("Id") for it in chg_items if it.get("Id")][:5]
if sample_ids:
    resp_many = items_many(client, sample_ids) 
    prod_items = resp_many.get("Items") or []
    print("\nMANY items:", len(prod_items))
    if prod_items:
        print("MANY first product:\n", json.dumps(prod_items[0], indent=2, ensure_ascii=False))
else:
    print("\nMANY: ei Id-näytteitä")



In [0]:
import re

URL_PAT = re.compile(r'^https?://', re.I)
EXT_PAT = re.compile(r'\.(jpg|jpeg|png|webp|gif|tif|tiff|pdf)$', re.I)
KEY_HINTS = {'url','uri','link','image','picture','media','asset','file','thumbnail','imageurl','imageuri'}

def walk(obj, path=""):
    if isinstance(obj, dict):
        for k, v in obj.items():
            kp = f"{path}.{k}" if path else k
            yield from walk(v, kp)
    elif isinstance(obj, list):
        for i, v in enumerate(obj):
            yield from walk(v, f"{path}[{i}]")
    else:
        yield path, obj

def print_image_candidates(item):
    print("\n=== Mahdolliset kuva/URL-kentät ===")
    found = False
    for p, v in walk(item):
        # osuma jos arvo näyttää URL:lta tai tiedostopäätteeltä
        if isinstance(v, str) and (URL_PAT.search(v) or EXT_PAT.search(v)):
            print(f"{p}: {v}")
            found = True
        # osuma jos polussa on “vihjeavaimia”
        else:
            low = p.lower()
            if any(h in low.split('.')[-1] for h in KEY_HINTS):
                if isinstance(v, (str, int, float)):
                    print(f"{p}: {v}")
                    found = True
    if not found:
        print("(Ei suoria URL-arvoja löytynyt.)")

# Kun sinulla on prod_items (items_many-vastauksesta):
if prod_items:
    # näytä ylimmän tason avaimet nopeaan silmäykseen
    print("Top-level keys:", sorted(prod_items[0].keys()))
    # etsi URL- ja kuvaehdokkaat
    print_image_candidates(prod_items[0])
